# ArXiv API — Searching and Fetching Papers

This notebook shows how to interact with ArXiv programmatically using the
[`arxiv`](https://pypi.org/project/arxiv/) Python package, which wraps the official ArXiv API.

You will learn how to:

- Search ArXiv by keyword, author, and category.
- Fetch a specific paper by its ArXiv identifier.

These are the same operations you would do manually on the ArXiv website — just scriptable,
so you can automate literature searches or build a personal pipeline.

## Setup

The `arxiv` package is included in Nestling's dependencies.
If you followed the [Environment Setup](../../../docs/getting-started/environment-setup.md),
it is already installed.

In [ ]:
import arxiv

## 1. Keyword Search

The `arxiv.Search` class takes a query string that supports:

- **Boolean operators**: `AND`, `OR`, `NOT`
- **Field prefixes**: `ti:` (title), `au:` (author), `abs:` (abstract), `cat:` (category)

Examples:
```
"ti:neutrino mass AND cat:hep-ph"
"au:Meighen-Berger AND cat:hep-ph"
"abs:dark matter AND cat:astro-ph.HE"
```

We sort by submission date so the most recent results come first.

In [ ]:
def search_arxiv(query: str, max_results: int = 5) -> list[arxiv.Result]:
    """Search ArXiv and return a list of Result objects."""
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
    )
    return list(client.results(search))


def print_result(r: arxiv.Result) -> None:
    """Pretty-print a single ArXiv result."""
    authors = ", ".join(str(a) for a in r.authors[:3])
    if len(r.authors) > 3:
        authors += " et al."
    arxiv_id = r.entry_id.split("/")[-1].split("v")[0]  # strip version suffix
    print(f"Title  : {r.title}")
    print(f"Authors: {authors}")
    print(f"Date   : {r.published.date()}")
    print(f"ArXiv  : {arxiv_id}")
    print()


results = search_arxiv("au:Meighen-Berger AND cat:hep-ph", max_results=5)
print(f"Found {len(results)} result(s)\n")
for r in results:
    print_result(r)

## 2. Fetching a Specific Paper by ArXiv ID

When you know the ArXiv identifier of a paper — for example from a reference list or a colleague's message —
you can fetch it directly without searching.

Here we use three recent papers from the authors of this repository as concrete benchmarks.
They cover atmospheric neutrinos, dark matter, and a neutrino telescope benchmark suite —
each from a different collaboration, which also illustrates that arXiv IDs are stable and
work regardless of which journal (or no journal) the paper ends up in.

In [ ]:
def fetch_paper(arxiv_id: str) -> arxiv.Result | None:
    """Fetch a single paper by its ArXiv identifier. Returns None if not found."""
    client = arxiv.Client()
    results = list(client.results(arxiv.Search(id_list=[arxiv_id])))
    return results[0] if results else None


# Three recent papers used as benchmarks throughout this notebook.
# (Bias note: these are HEP papers; the API works identically for any field.)
BENCHMARK_PAPERS = [
    ("2605.16721", "Beacom et al. — CP-violating phase, atmospheric neutrinos"),
    ("2512.18093", "Meighen-Berger et al. — dark matter & ultrahigh-energy cosmic rays"),
    ("2511.13111", "Orsoe et al. — NuBench, deep-learning benchmark for neutrino telescopes"),
]

for arxiv_id, description in BENCHMARK_PAPERS:
    paper = fetch_paper(arxiv_id)
    if paper:
        print(f"[{description}]")
        print_result(paper)
    else:
        print(f"[{description}]")
        print(f"  arXiv:{arxiv_id} not found (paper may not yet be indexed)\n")

## 3. Accessing Metadata and the PDF

Each `arxiv.Result` object carries much more than just the title.
Here is a quick overview of the fields you are most likely to use.

In [ ]:
paper = fetch_paper("2512.18093")

if paper:
    print("Title       :", paper.title)
    print("Authors     :", [str(a) for a in paper.authors])
    print("Published   :", paper.published)
    print("Abstract    :", paper.summary[:300], "...")
    print("Categories  :", paper.categories)
    print("PDF URL     :", paper.pdf_url)
    print("Inspire link: https://inspirehep.net/search?p=eprint+" + "2512.18093")

## 4. Try It Yourself

Replace the query or the ArXiv ID below with anything relevant to your project.
The API is free and does not require authentication.

In [ ]:
# Change this to a topic relevant to your research
MY_QUERY = "ti:dark matter AND cat:hep-ph"
MY_MAX = 5

results = search_arxiv(MY_QUERY, max_results=MY_MAX)
print(f"Top {len(results)} result(s) for query: '{MY_QUERY}'\n")
for r in results:
    print_result(r)